# PD06 · PQRS completas: agregaciones de alta cardinalidad en Polars CPU y GPU
Curso de Postgrado: BigData, Especialización en Bases de datos · CyberColombia.

**Plantilla individual por desarrollar.** Copia este notebook a `soluciones/`
con el mismo nombre. Completa código, controles e interpretación; no se entrega
una solución ni resultados medidos. La GPU y su kernel se consideran verificados.

Compararás **Polars CPU frente a Polars GPU**, con todos los datos de tres
periodos de Supersalud, sin pandas para las consultas. Resolverás dos ejercicios:

1. Departamentos con más quejas según edad, patología, motivo y canal.
2. Evolución de esas mismas combinaciones mes a mes, en orden cronológico ascendente.

Las claves de texto y el número de grupos aumentan el trabajo de agregación y
ordenación. Debes medirlo: no se garantiza saturar la GPU ni obtener aceleración.
No copies filas para simular más datos. Registra la cardinalidad observada.


## 1. Preparar los tres CSV completos

Desde una terminal Ubuntu/WSL, en la raíz del repositorio y con el entorno activo:

```bash
python kit/00_datos.py --descargar pqrs-completos
python kit/00_datos.py --verificar pqrs-completos
```

Deben quedar en `kit/data/pqrs_completos/`:

| Corte | CSV | Reportes de referencia |
|---|---|---:|
| 2023-II | `PQRD_2023_II.csv` | 716.697 |
| 2024-I | `PQRD_2024_I.csv` | 781.601 |
| 2024-II | `PQRD_2024_II.csv` | 946.468 |

Total: **2.444.766 reportes**, aproximadamente 1,82 GB de CSV. El descargador
conserva archivos correctos y verifica los tres. No son muestras.
No ejecutes esos comandos dentro del reloj del benchmark.


In [1]:
import polars as pl
from polars.testing import assert_frame_equal

consulta = pl.LazyFrame({"valor": [1, 2, 3]}).select(pl.col("valor").sum())
cpu = consulta.collect(engine="in-memory")
gpu = consulta.collect(engine=pl.GPUEngine(raise_on_fail=True))
assert gpu.item() == 6
assert_frame_equal(cpu, gpu)
print("Consulta GPU correcta:", gpu)

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

You should always run with libnvidia-ml.so that is installed with your
NVIDIA Display Driver. By default it's installed in /usr/lib and /usr/lib64.

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

You should always run with libnvidia-ml.so that is installed with your
NVIDIA Display Driver. By default it's installed in /usr/lib and /usr/lib64.

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!



RuntimeError: RapidsMPF fatal error at: /__w/rapidsmpf/rapidsmpf/cpp/src/system_info.cpp:130: No GPUs found on current NUMA node 0

In [ ]:
from pathlib import Path
from time import perf_counter
from importlib import metadata
import json, platform, sys
import polars as pl
from polars.testing import assert_frame_equal
from IPython.display import display

REPOSITORIO = None  # Ejemplo: Path('/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate')
if REPOSITORIO is None:
    REPOSITORIO = next((p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / 'kit/pqrs_fuentes.json').is_file()), None)
assert REPOSITORIO is not None, 'Indicar la ruta del repositorio'
REPOSITORIO = Path(REPOSITORIO).expanduser().resolve()
KIT = REPOSITORIO / 'kit'
CSV_DIR = KIT / 'data/pqrs_completos'
OUT = KIT / 'salidas/soluciones/PD06'
PARQUET_DIR = OUT / 'parquet'
PARQUET_DIR.mkdir(parents=True, exist_ok=True)
PERIODOS = ['2023_II', '2024_I', '2024_II']
ARCHIVOS_CSV = [CSV_DIR / f'PQRD_{periodo}.csv' for periodo in PERIODOS]
ARCHIVOS_PARQUET = [PARQUET_DIR / f'{periodo}.parquet' for periodo in PERIODOS]
FILAS_ESPERADAS = [716697, 781601, 946468]
TOTAL_ESPERADO = sum(FILAS_ESPERADAS)
REPETICIONES = 3
TOP_N = 10
DISPOSITIVO_CUDA = 0
assert all(p.is_file() for p in ARCHIVOS_CSV), 'Completar descarga y verificación'
assert REPETICIONES >= 3
print('Python:', sys.executable)
print('Polars:', pl.__version__)
print('Entrada:', CSV_DIR)
print('Resultados:', OUT)


## 2. Leer y convertir con Polars · desarrollo del estudiante

Completa la siguiente celda sin pandas ni funciones del taller que oculten la conversión:

1. Inspecciona cinco filas de cada CSV con `pl.read_csv`. Usa `;`, `,` y `;`,
   respectivamente. Conserva códigos y las 38 columnas originales como texto.
2. Usa `pl.scan_csv` y transforma con expresiones Polars. Añade:
   - `corte_archivo`: corte del CSV de origen.
   - `codigo_departamento`: desde `afec_cod_depto`, dos dígitos; marca inválidos.
   - `edad_anios`: desde `afec_edad`, numérica. Explicita el tratamiento de edades
     vacías, negativas, no numéricas o no finitas; conserva estos reportes en un grupo nulo.
   - `anio` y `mes_num`: números desde `periodo` y `mes`, no desde el nombre semestral.
3. Escribe un Parquet por corte con `sink_parquet`, Zstandard y streaming CPU.
4. Comprueba que están los tres, que conservan sus filas y el contrato de columnas.
5. Al reejecutar, comprueba los Parquet existentes antes de decidir reutilizarlos.

La preparación se hace con Polars CPU; la GPU se utilizará para las agregaciones.
No mezcles este tiempo con los de consulta. Guarda el tiempo de conversión aparte.


In [ ]:
# TODO: inspeccionar los tres CSV y desarrollar aquí su conversión con Polars.
# Operaciones esperadas: read_csv, scan_csv, with_columns y sink_parquet.
# No llamar a un notebook resuelto ni usar pandas para esta preparación.
raise NotImplementedError('Desarrollar lectura, normalización y conversión con Polars')


In [ ]:
# Control de preparación; ejecutar después de completar la celda anterior.
assert all(p.is_file() for p in ARCHIVOS_PARQUET), 'Faltan periodos convertidos'
filas_por_archivo = [pl.scan_parquet(p).select(pl.len()).collect().item()
                     for p in ARCHIVOS_PARQUET]
assert filas_por_archivo == FILAS_ESPERADAS
assert sum(filas_por_archivo) == TOTAL_ESPERADO
REQUERIDAS = ['codigo_departamento', 'edad_anios', 'patologia_1',
              'cod_motesp', 'motivo_especifico', 'pqr_canal', 'anio', 'mes_num']
for archivo in ARCHIVOS_PARQUET:
    esquema = pl.scan_parquet(archivo).collect_schema()
    assert set(REQUERIDAS) <= set(esquema.names())
    assert esquema['anio'].is_integer() and esquema['mes_num'].is_integer()
    assert esquema['edad_anios'].is_numeric()
print('Tres periodos conservados:', filas_por_archivo)


## 3. Ejercicio 1 · Departamentos con más quejas por perfil

**Combina todos los periodos completos**, no solo una muestra. Define el perfil:

| Dimensión | Campo |
|---|---|
| Edad en años | `edad_anios`, construida desde `afec_edad` |
| Patología | `patologia_1` |
| Código del motivo específico | `cod_motesp` |
| Descripción del mismo motivo | `motivo_especifico` |
| Canal de entrada | `pqr_canal` |

No mezcles un código de motivo específico con la descripción de otro nivel.
Agrupa por departamento y las cinco dimensiones; cuenta filas como `quejas`.
Ordena por perfil, quejas descendentes y código de departamento ascendente
para resolver empates de forma reproducible. Conserva los grupos no informados.

**Producto:** una tabla detallada con todas las combinaciones y sus conteos;
y una vista de hasta `TOP_N` departamentos con más quejas **para cada perfil**.
No basta un único top nacional que ignore edad, patología, motivo y canal.
Puedes preparar ese recorte después de medir, indicándolo en el informe.

El plan medido debe incluir lectura Parquet, agrupación y ordenamiento.
Declara un `LazyFrame` llamado `consulta_e1`; no lo ejecutes todavía.
No presentes quejas como personas únicas ni como tasas por población.


In [ ]:
# TODO: desarrollar las expresiones y la consulta LazyFrame del ejercicio 1.
# Deben verse select, group_by, agg y sort.
consulta_e1 = None
raise NotImplementedError('Desarrollar la agregación y ordenación del ejercicio 1')


## 4. Ejercicio 2 · Evolución mensual de los mismos perfiles

Conserva departamento, edad, patología, código y descripción del motivo y canal.
Añade **año y mes numéricos** a la agrupación, cuenta las quejas y ordena el
resultado **primero por año y mes ascendentes**, después por las dimensiones
para que la salida sea reproducible. Enero, febrero y octubre deben quedar
como meses 1, 2 y 10, no como cadenas ordenadas alfabéticamente.

**Producto:** tabla `anio`, `mes_num`, departamento, las cinco dimensiones y
`quejas`, que permita seguir un mismo perfil mes a mes. Muestra dos perfiles
concretos y explica sus cambios. No concluyas que toda variación es tendencia.

Conserva las fechas inválidas en el control total y sepáralas al presentar la
cronología. No inventes meses ni rellenes silenciosamente ausencias con cero.
Declara el plan lazy como `consulta_e2`, sin ejecutarlo todavía.


In [ ]:
# TODO: desarrollar el ejercicio 2: mismas dimensiones + anio y mes_num.
# La ordenación por anio y mes_num debe estar en el plan medido.
consulta_e2 = None
raise NotImplementedError('Desarrollar la agregación mensual y su orden ascendente')


## 5. Contrato de comparación y frontera temporal

Completa las listas de claves y la comprobación para cada resultado.
Verifica todas las filas agregadas, no solo la suma global. Los conteos deben
ser exactos y cada consulta debe conservar 2.444.766 reportes, incluidos grupos
no informados. Revisa tipos, nulos y duplicados de claves agregadas.

Mediremos con `perf_counter`: **leer Parquet + agregar + ordenar + devolver un
DataFrame Polars en CPU**. `collect` termina esa frontera para ambos motores.
No se mide el tiempo de construir el plan, descargar, convertir, verificar ni
exportar. La medición GPU incluye transferencias; no es tiempo exclusivo de
kernels CUDA ni una comparación de datos ya residentes en VRAM.

CPU usará `engine='streaming'`. GPU usará `GPUEngine(raise_on_fail=True)`.
Si el plan no está soportado en GPU, corrígelo o informa el error; no lo midas
como GPU mediante fallback a CPU. No cambies filtros o claves entre motores.


In [ ]:
assert isinstance(consulta_e1, pl.LazyFrame), 'Completar ejercicio 1'
assert isinstance(consulta_e2, pl.LazyFrame), 'Completar ejercicio 2'
CONSULTAS = {'e1_departamentos': consulta_e1, 'e2_evolucion_mensual': consulta_e2}
CLAVES_RESULTADO = {
    'e1_departamentos': [],  # TODO: claves completas, sin la columna quejas
    'e2_evolucion_mensual': []
}
assert all(CLAVES_RESULTADO.values()), 'Definir las claves para comparar todos los grupos'
engine_gpu = pl.GPUEngine(device=DISPOSITIVO_CUDA, raise_on_fail=True)

# TODO: comprobar esquema, unicidad de claves, nulos y total de cada resultado.
# Mantén visibles las comprobaciones en el notebook antes de interpretar tiempos.
primeras = []


### Ejercicio 1 · Primera ejecución en CPU y GPU

Ejecuta la celda después de completar los planes. La primera llamada puede
incluir inicialización y cachés; se guarda por separado de las repeticiones.


In [ ]:
# CPU: se cronometra la ejecución completa del plan, no su declaración.
inicio = perf_counter()
cpu_e1 = consulta_e1.collect(engine='streaming')
tiempo_cpu_e1 = perf_counter() - inicio

# GPU: mismos datos, claves, agregación y orden.
inicio = perf_counter()
gpu_e1 = consulta_e1.collect(engine=engine_gpu)
tiempo_gpu_e1 = perf_counter() - inicio

# Fuera del reloj: comparar todos los grupos y conservar el total.
assert_frame_equal(cpu_e1.sort(CLAVES_RESULTADO['e1_departamentos']),
                   gpu_e1.sort(CLAVES_RESULTADO['e1_departamentos']), check_exact=True)
assert cpu_e1['quejas'].sum() == TOTAL_ESPERADO
primeras.extend([
    {'ejercicio': 'e1', 'motor': 'cpu', 'segundos': tiempo_cpu_e1},
    {'ejercicio': 'e1', 'motor': 'gpu', 'segundos': tiempo_gpu_e1}])
print('Ejercicio 1 · CPU:', tiempo_cpu_e1, 's · GPU:', tiempo_gpu_e1, 's')
display(gpu_e1.head(20))


### Ejercicio 2 · Primera ejecución en CPU y GPU

Ejecuta la celda después de completar los planes. La primera llamada puede
incluir inicialización y cachés; se guarda por separado de las repeticiones.


In [ ]:
# CPU: se cronometra la ejecución completa del plan, no su declaración.
inicio = perf_counter()
cpu_e2 = consulta_e2.collect(engine='streaming')
tiempo_cpu_e2 = perf_counter() - inicio

# GPU: mismos datos, claves, agregación y orden.
inicio = perf_counter()
gpu_e2 = consulta_e2.collect(engine=engine_gpu)
tiempo_gpu_e2 = perf_counter() - inicio

# Fuera del reloj: comparar todos los grupos y conservar el total.
assert_frame_equal(cpu_e2.sort(CLAVES_RESULTADO['e2_evolucion_mensual']),
                   gpu_e2.sort(CLAVES_RESULTADO['e2_evolucion_mensual']), check_exact=True)
assert cpu_e2['quejas'].sum() == TOTAL_ESPERADO
primeras.extend([
    {'ejercicio': 'e2', 'motor': 'cpu', 'segundos': tiempo_cpu_e2},
    {'ejercicio': 'e2', 'motor': 'gpu', 'segundos': tiempo_gpu_e2}])
print('Ejercicio 2 · CPU:', tiempo_cpu_e2, 's · GPU:', tiempo_gpu_e2, 's')
display(gpu_e2.head(20))


## 6. Calentamiento y tres repeticiones por motor

Completa el bucle de medición. Por cada ejercicio:

1. Ejecuta una vuelta adicional de calentamiento CPU/GPU y no la mezcles con las medidas.
2. Repite al menos tres veces, alternando CPU→GPU y GPU→CPU.
3. Rodea cada `collect` con `perf_counter`; compara resultados después de parar el reloj.
4. Guarda `ejercicio`, `motor`, `repeticion`, `segundos`, `filas_entrada`,
   `grupos_resultado` y `equivalente` en la lista `mediciones`.
5. Revisa que los resultados conserven todos los reportes y las claves esperadas.

Mide los mismos planes usados arriba. Mantén las mismas condiciones y registra
hilos CPU, GPU, VRAM, controlador y versiones. No se borran automáticamente las
cachés del sistema operativo; explica su efecto. No uses `process_time` como
sustituto de la duración total de la consulta GPU.


In [ ]:
mediciones = []
# TODO: desarrollar aquí calentamiento, alternancia, reloj, validación y registro.
# La ejecución de cada collect debe quedar visible en este bucle.
raise NotImplementedError('Completar las tres repeticiones CPU/GPU de ambos ejercicios')


## 7. Tabla de tiempos y factor de aceleración

La celda siguiente no resuelve consultas: resume tus mediciones. Habrá
**12 filas como mínimo**: dos ejercicios × dos motores × tres repeticiones.
Factor CPU/GPU = mediana CPU / mediana GPU: mayor que 1 favorece GPU;
menor que 1 favorece CPU. No mezcles ambas consultas ni las primeras ejecuciones.


In [ ]:
assert len(mediciones) == len(CONSULTAS) * 2 * REPETICIONES
requeridas = {'ejercicio', 'motor', 'repeticion', 'segundos',
              'filas_entrada', 'grupos_resultado', 'equivalente'}
assert all(requeridas <= set(fila) for fila in mediciones)
tiempos = pl.DataFrame(mediciones)
assert tiempos['equivalente'].all()
assert (tiempos['filas_entrada'] == TOTAL_ESPERADO).all()
assert tiempos['segundos'].is_finite().all() and (tiempos['segundos'] > 0).all()
assert set(tiempos['ejercicio'].to_list()) == set(CONSULTAS)
assert set(tiempos['motor'].to_list()) == {'cpu', 'gpu'}
conteos = tiempos.group_by(['ejercicio', 'motor']).agg(pl.len().alias('n'))
assert (conteos['n'] == REPETICIONES).all()
assert tiempos.select(['ejercicio', 'motor', 'repeticion']).unique().height == tiempos.height
resumen = tiempos.group_by(['ejercicio', 'motor']).agg(
    pl.col('segundos').median().alias('mediana_s'),
    pl.col('segundos').min().alias('min_s'),
    pl.col('segundos').max().alias('max_s')).sort(['ejercicio', 'motor'])
factores = []
for nombre in CONSULTAS:
    cpu_s = resumen.filter((pl.col('ejercicio') == nombre) & (pl.col('motor') == 'cpu'))['mediana_s'].item()
    gpu_s = resumen.filter((pl.col('ejercicio') == nombre) & (pl.col('motor') == 'gpu'))['mediana_s'].item()
    factores.append({'ejercicio': nombre, 'mediana_cpu_s': cpu_s,
                     'mediana_gpu_s': gpu_s, 'factor_cpu_sobre_gpu': cpu_s / gpu_s})
display(resumen)
display(pl.DataFrame(factores))
tiempos.write_csv(OUT / 'benchmark_PD06.csv')
resumen.write_csv(OUT / 'resumen_PD06.csv')
pl.DataFrame(primeras).write_csv(OUT / 'primeras_ejecuciones_PD06.csv')
pl.DataFrame(factores).write_csv(OUT / 'factores_PD06.csv')


## 8. Interpretar y entregar

Completa el notebook y entrega en tu repositorio:

- `soluciones/PD06_PQRS_agregaciones_CPU_GPU.ipynb` con código visible y conclusiones.
- Resultados regenerables en `kit/salidas/soluciones/PD06/`: detalle/ranking de
  departamentos, evolución mensual, controles, tiempos, resumen y factores.
- Un control JSON con filas por corte, total conciliado, número de grupos por
  consulta, reglas de edad y fechas, cobertura de departamentos, versiones,
  modelo GPU/VRAM/controlador, hilos CPU y alcance de cada medición.

**Responde:** ¿qué departamentos lideran dentro de cada perfil? ¿Cómo evolucionan
los dos perfiles seleccionados? ¿Cuántos grupos genera cada consulta? ¿La GPU
acelera ambas? ¿Qué parte puede estar limitada por lectura, transferencias,
ordenamiento o memoria? ¿Cambió la conclusión entre primera ejecución y mediana?

Se evalúan corrección, equivalencia, medición reproducible e interpretación;
no que GPU necesariamente gane. No adjuntes los CSV originales al repositorio.
La salida queda en la ruta ignorada `kit/salidas/`. La copia resuelta pertenece
al estudiante; esta plantilla mantiene sus celdas por completar.

Apoyo: [guía Polars](../docs/manual-polars.md),
[entorno GPU](../docs/polars-gpu.md),
[referencia P04](../talleres/P04.md). Son referencias, no una solución de PD06.
